In [3]:
import py3Dmol

# File stays on the cluster — only coordinates are sent to the browser
with open("/cluster/project/beltrao/kdammer/master_thesis/tmp/p00431_p00431_pool_0.pdb", "r") as f:
    pdb_data = f.read()

view = py3Dmol.view(width=600, height=400)
view.addModel(pdb_data, "pdb")
view.setStyle({"cartoon": {"color": "blue"}})          # PyMOL-like cartoon
view.setStyle({"hetflag": True}, {"stick": {}})            # ligands as sticks
view.setStyle({"bonds": 0}, {"sphere": {"radius": 0.5}})   # ions/waters as spheres
view.zoomTo()
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [2]:
import py3Dmol
from Bio.PDB import PDBParser, Superimposer, PDBIO

ref_path  = "/cluster/project/beltrao/kdammer/master_thesis/tmp/p00431_p00431_pair_0.pdb"
mob_path  = "/cluster/project/beltrao/kdammer/master_thesis/tmp/p00431_p00431_pool_0.pdb"
aligned_path = "/cluster/project/beltrao/kdammer/master_thesis/tmp/pool_aligned.pdb"

# --- 1. Parse both structures ---
parser = PDBParser(QUIET=True)
ref = parser.get_structure("ref", ref_path)
mob = parser.get_structure("mob", mob_path)

# --- 2. Match CA atoms by (chain, residue id) ---
ref_ca  = {a.get_parent().get_id(): a for a in ref.get_atoms()  if a.name == "CA"}
mob_ca  = {a.get_parent().get_id(): a for a in mob.get_atoms()  if a.name == "CA"}

common = sorted(set(ref_ca.keys()) & set(mob_ca.keys()))
ref_atoms  = [ref_ca[r]  for r in common]
mob_atoms  = [mob_ca[r]  for r in common]

print(f"Aligning on {len(common)} CA atoms")

# --- 3. Superimpose mob onto ref ---
sup = Superimposer()
sup.set_atoms(ref_atoms, mob_atoms)
sup.apply(list(mob.get_atoms()))   # transform ALL atoms, not just CA

print(f"RMSD: {sup.rms:.2f} Å")

# --- 4. Save aligned mobile structure ---
io = PDBIO()
io.set_structure(mob)
io.save(aligned_path)

# --- 5. Display both overlaid in py3Dmol ---
with open(ref_path, "r") as f:
    ref_data = f.read()
with open(aligned_path, "r") as f:
    mob_data = f.read()

view = py3Dmol.view(width=700, height=500)

# Reference (pair) — blue
view.addModel(ref_data, "pdb")
view.setStyle({"model": 0}, {"cartoon": {"color": "blue"}})

# Mobile (pool, aligned) — magenta
view.addModel(mob_data, "pdb")
view.setStyle({"model": 1}, {"cartoon": {"color": "magenta"}})

# Ligands as sticks in both
view.setStyle({"model": 0, "hetflag": True}, {"stick": {"colorscheme": "blueCarbon"}})
view.setStyle({"model": 1, "hetflag": True}, {"stick": {"colorscheme": "magentaCarbon"}})

view.zoomTo()
view.show()

Aligning on 361 CA atoms
RMSD: 19.31 Å


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
import numpy as np
import py3Dmol
from Bio.PDB import PDBParser, Superimposer, PDBIO

ref_path = "/cluster/project/beltrao/kdammer/master_thesis/tmp/p00431_p00431_pair_0.pdb"
mob_path = "/cluster/project/beltrao/kdammer/master_thesis/tmp/p00431_p00431_pool_0.pdb"
aligned_path = "/cluster/project/beltrao/kdammer/master_thesis/tmp/pool_aligned.pdb"

parser = PDBParser(QUIET=True)
ref = parser.get_structure("ref", ref_path)
mob = parser.get_structure("mob", mob_path)

# --- Match CA atoms by (chain, resseq) ---
ref_ca = {(a.get_parent().get_parent().id, a.get_parent().id[1]): a
          for a in ref.get_atoms() if a.name == "CA"}
mob_ca = {(a.get_parent().get_parent().id, a.get_parent().id[1]): a
          for a in mob.get_atoms() if a.name == "CA"}

common = sorted(set(ref_ca.keys()) & set(mob_ca.keys()))
ref_atoms = [ref_ca[r] for r in common]
mob_atoms = [mob_ca[r] for r in common]

print(f"Matched {len(common)} CA atoms")
print(f"Ref chains: {sorted(set(k[0] for k in ref_ca))}")
print(f"Mob chains: {sorted(set(k[0] for k in mob_ca))}")

# --- Iterative outlier rejection (mimics PyMOL align) ---
def align_with_rejection(ref_atoms, mob_atoms, max_cycles=5, reject_factor=2.0):
    ref_list = list(ref_atoms)
    mob_list = list(mob_atoms)
    sup = Superimposer()

    for cycle in range(max_cycles):
        sup.set_atoms(ref_list, mob_list)
        sup.apply(mob_list)

        devs = np.array([np.linalg.norm(r.coord - m.coord)
                         for r, m in zip(ref_list, mob_list)])
        rmsd = sup.rms
        n = len(ref_list)
        threshold = reject_factor * rmsd
        keep = devs < threshold

        print(f"Cycle {cycle+1}: {n} atoms, RMSD={rmsd:.2f} Å, "
              f"threshold={threshold:.1f} Å, rejected={int((~keep).sum())}")

        if keep.all() or n <= 10:
            break
        ref_list = [a for a, k in zip(ref_list, keep) if k]
        mob_list = [a for a, k in zip(mob_list, keep) if k]

    # Apply final transform to ALL atoms in the mobile structure
    sup.set_atoms(ref_list, mob_list)
    sup.apply(list(mob.get_atoms()))
    return sup

sup = align_with_rejection(ref_atoms, mob_atoms)
print(f"Final RMSD: {sup.rms:.2f} Å")

# --- Save + visualize ---
io = PDBIO()
io.set_structure(mob)
io.save(aligned_path)

with open(ref_path) as f:
    ref_data = f.read()
with open(aligned_path) as f:
    mob_data = f.read()

view = py3Dmol.view(width=700, height=500)
view.addModel(ref_data, "pdb")
view.setStyle({"model": 0}, {"cartoon": {"color": "blue"}})
view.addModel(mob_data, "pdb")
view.setStyle({"model": 1}, {"cartoon": {"color": "magenta"}})
view.zoomTo()
view.show()

Matched 722 CA atoms
Ref chains: ['A', 'B']
Mob chains: ['A', 'B']
Cycle 1: 722 atoms, RMSD=41.61 Å, threshold=83.2 Å, rejected=35
Cycle 2: 687 atoms, RMSD=35.32 Å, threshold=70.6 Å, rejected=13
Cycle 3: 674 atoms, RMSD=33.81 Å, threshold=67.6 Å, rejected=11
Cycle 4: 663 atoms, RMSD=32.54 Å, threshold=65.1 Å, rejected=8
Cycle 5: 655 atoms, RMSD=31.50 Å, threshold=63.0 Å, rejected=2
Final RMSD: 31.31 Å


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [8]:
from Bio.PDB import PDBParser, Superimposer

# 1. Pfade definieren
ref_path = "/cluster/project/beltrao/kdammer/master_thesis/tmp/p00431_p00431_pair_0.pdb"
mob_path = "/cluster/project/beltrao/kdammer/master_thesis/tmp/p00431_p00431_pool_0.pdb"

# 2. Strukturen einlesen
parser = PDBParser(QUIET=True)
ref_struct = parser.get_structure("reference", ref_path)
mob_struct = parser.get_structure("mobile", mob_path)

# 3. CA-Atome der ersten Modelle extrahieren
ref_atoms = [atom for atom in ref_struct[0].get_atoms() if atom.get_name() == "CA"]
mob_atoms = [atom for atom in mob_struct[0].get_atoms() if atom.get_name() == "CA"]

# 4. Überprüfung der Atom-Anzahl (Biopython benötigt exakt gleich viele Atome)
if len(ref_atoms) != len(mob_atoms):
    print(f"Warnung: Unterschiedliche Anzahl an CA-Atomen! Ref: {len(ref_atoms)}, Mob: {len(mob_atoms)}")
    # Kürzen auf die minimale gemeinsame Länge (falls die Sequenzen identisch nummeriert sind)
    min_len = min(len(ref_atoms), len(mob_atoms))
    ref_atoms = ref_atoms[:min_len]
    mob_atoms = mob_atoms[:min_len]

# 5. Superimposer initialisieren und berechnen
super_imposer = Superimposer()
super_imposer.set_atoms(ref_atoms, mob_atoms)

# 6. Ausrichtung auf die mobile Struktur anwenden (optional)
super_imposer.apply(mob_struct.get_atoms())

# 7. RMSD ausgeben
print(f"✅ Berechneter RMSD: {super_imposer.rms:.4f} Å")


✅ Berechneter RMSD: 41.6128 Å
